# Netflix Content Analytics — Coding Challenge

## Scenario

You're a data scientist on Netflix's Content Analytics team. The product team ran an initiative in Q1 2026 aimed at improving how often users complete the content they start watching.

You have a dataset of ~215,000 viewing sessions across Q4 2025 and Q1 2026. Explore the data and evaluate whether the initiative improved completion rates.

---
**Dataset:** generated by the setup cell below — nothing to download.  
**Key columns:** `quarter`, `watch_time_minutes`, `completed`, `genre`, `device_type`, `content_type`, `user_age_group`, `subscription_tier`

In [ ]:
# ---------------------------------------------------------------------------
# Setup. Running this cell generates the dataset — nothing to download.
# Synthetic data, seeded, so everyone gets identical numbers.
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

import pandas as pd
np.random.seed(42)
# ---------------------------------------------------------------------------
# Data parameters
# ---------------------------------------------------------------------------
SESSIONS = {
    "Q4_2025": 100_000,
    "Q1_2026": 115_000,
}
# Genre mix: Documentary jumps from 5% → 35% in Q1 (Netflix pushed non-fiction content).
# This mix shift is the root cause of the aggregate paradox.
GENRE_DIST = {
    "Q4_2025": {"Drama": 0.30, "Comedy": 0.25, "Action": 0.25, "Romance": 0.15, "Documentary": 0.05},
    "Q1_2026": {"Drama": 0.20, "Comedy": 0.18, "Action": 0.17, "Romance": 0.10, "Documentary": 0.35},
}
# Every genre improves — the product initiative genuinely worked at the genre level.
# But Documentary's baseline rate (43–46%) drags down the aggregate when it dominates Q1.
COMPLETION_RATES = {
    "Drama":       {"Q4_2025": 0.72, "Q1_2026": 0.74},
    "Comedy":      {"Q4_2025": 0.71, "Q1_2026": 0.73},
    "Action":      {"Q4_2025": 0.75, "Q1_2026": 0.76},
    "Romance":     {"Q4_2025": 0.70, "Q1_2026": 0.72},
    "Documentary": {"Q4_2025": 0.43, "Q1_2026": 0.46},
}
# Documentaries are much longer on average (~95 min vs ~50 min for others),
# which also drags down aggregate average watch time despite per-genre improvement.
WATCH_TIME_PARAMS = {
    "Drama":       {"mean": 48, "std": 14},
    "Comedy":      {"mean": 52, "std": 16},
    "Action":      {"mean": 55, "std": 17},
    "Romance":     {"mean": 50, "std": 15},
    "Documentary": {"mean": 95, "std": 29},
}
DEVICE_TYPES  = ["TV", "Mobile", "Tablet", "Desktop"]
DEVICE_PROBS  = [0.50, 0.30, 0.10, 0.10]
CONTENT_TYPES = ["Movie", "Series"]
CONTENT_PROBS = [0.40, 0.60]
AGE_GROUPS    = ["18-24", "25-34", "35-44", "45-54", "55+"]
SUB_TIERS     = ["Basic", "Standard", "Premium"]
SUB_PROBS     = [0.30, 0.50, 0.20]
# ---------------------------------------------------------------------------
# Generate rows
# ---------------------------------------------------------------------------
rows = []
session_counter = 1
for quarter, n_sessions in SESSIONS.items():
    genres      = list(GENRE_DIST[quarter].keys())
    genre_probs = list(GENRE_DIST[quarter].values())
    genre_arr   = np.random.choice(genres, size=n_sessions, p=genre_probs)
    device_arr  = np.random.choice(DEVICE_TYPES, size=n_sessions, p=DEVICE_PROBS)
    content_arr = np.random.choice(CONTENT_TYPES, size=n_sessions, p=CONTENT_PROBS)
    age_arr     = np.random.choice(AGE_GROUPS, size=n_sessions)
    tier_arr    = np.random.choice(SUB_TIERS, size=n_sessions, p=SUB_PROBS)
    # ~10 sessions per user on average
    n_users        = n_sessions // 10
    user_arr       = np.random.randint(1, n_users + 1, size=n_sessions)
    content_id_arr = np.random.randint(1, 5001, size=n_sessions)  # pool of 5,000 titles
    for i in range(n_sessions):
        genre           = genre_arr[i]
        completion_rate = COMPLETION_RATES[genre][quarter]
        completed       = int(np.random.rand() < completion_rate)
        params     = WATCH_TIME_PARAMS[genre]
        watch_time = round(max(1.0, np.random.normal(params["mean"], params["std"])), 1)
        rows.append({
            "session_id":         session_counter,
            "user_id":            f"u_{user_arr[i]:06d}",
            "content_id":         f"c_{content_id_arr[i]:05d}",
            "quarter":            quarter,
            "watch_time_minutes": watch_time,
            "completed":          completed,
            "device_type":        device_arr[i],
            "genre":              genre,
            "content_type":       content_arr[i],
            "user_age_group":     age_arr[i],
            "subscription_tier":  tier_arr[i],
        })
        session_counter += 1
# ---------------------------------------------------------------------------
# Save and summarize
# ---------------------------------------------------------------------------
df = pd.DataFrame(rows)

df = pd.DataFrame(rows)
print(df.shape)
df.head()


---

## Question 1: Headline metrics

Calculate the following by quarter:
- Overall **completion rate**
- Average and total **watch time**

What story do these top-line numbers tell? Does anything surprise you?

In [ ]:
# Completion rate by quarter
df.groupby('quarter')['completed'].mean().round(4)

In [ ]:
# Watch time by quarter
df.groupby('quarter')['watch_time_minutes'].agg(
    avg_minutes='mean',
    total_hours=lambda x: x.sum() / 60
).round(1)

---

## Question 2: Break it down by genre

The headline metric may be hiding something. Explore completion rate and genre mix at the genre level:

- What is the completion rate **per genre** for each quarter? Did individual genres improve or decline?
- How did the **genre mix** (share of sessions per genre) shift between quarters?
- What does this tell you about why the aggregate number moved the way it did?

In [ ]:
# Completion rate by genre and quarter
completion_by_genre = (
    df.groupby(['genre', 'quarter'])['completed']
    .mean().round(4)
    .unstack()
)

completion_by_genre.sort_values('Q4_2025')

In [ ]:
# Genre mix (share of sessions) by quarter
genre_mix = (
    df.groupby('quarter')['genre']
    .value_counts(normalize=True)
    .mul(100).round(1)
    .rename('pct')
    .reset_index()
    .pivot(index='genre', columns='quarter', values='pct')
)
genre_mix['change'] = (genre_mix['Q1_2026'] - genre_mix['Q4_2025']).round(1)
genre_mix.sort_values('change')

---

## Question 3: Visualize the story

Build charts that together tell the Simpson's Paradox story clearly enough for a non-technical stakeholder (e.g., a PM or VP):

1. **Overall completion rate by quarter** — the misleading headline
2. **Line chart: completion rate by genre** — every genre trended up
3. **Grouped bar: completion rate by genre and quarter** — the per-genre improvement
4. **Genre mix shift** — what actually changed between quarters

What would you say in a slide or readout to explain why the headline metric went down even though the product initiative worked?

In [ ]:
# Chart 1: Overall completion rate by quarter
quarter_order = ['Q4_2025', 'Q1_2026']
overall = df.groupby('quarter')['completed'].mean().reindex(quarter_order).reset_index()

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=overall, x='quarter', y='completed', order=quarter_order,
            hue='quarter', palette='Blues_d', legend=False, ax=ax)
ax.set_ylim(0.60, 0.75)
ax.set_title('Overall Completion Rate by Quarter', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Completion Rate')
for bar, val in zip(ax.patches, overall['completed']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Completion rate by genre over time (line chart)
quarter_order = ['Q4_2025', 'Q1_2026']
genre_comp = (
    df.groupby(['quarter', 'genre'])['completed']
    .mean().round(4)
    .reset_index()
)
genre_comp['quarter'] = pd.Categorical(genre_comp['quarter'], categories=quarter_order, ordered=True)
genre_comp = genre_comp.sort_values('quarter')

fig, ax = plt.subplots(figsize=(8, 5))
for genre, grp in genre_comp.groupby('genre'):
    ax.plot(grp['quarter'], grp['completed'], marker='o', linewidth=2, label=genre)
    ax.annotate(f"{grp['completed'].iloc[-1]:.0%}",
                xy=(grp['quarter'].iloc[-1], grp['completed'].iloc[-1]),
                xytext=(5, 0), textcoords='offset points', va='center', fontsize=9)

ax.set_ylim(0.35, 0.85)
ax.set_title('Completion Rate by Genre (Q4 2025 → Q1 2026)', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Completion Rate')
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Genre mix shift between quarters
quarter_order = ['Q4_2025', 'Q1_2026']
mix = (
    df.groupby('quarter')['genre']
    .value_counts(normalize=True)
    .mul(100).round(1)
    .rename('pct')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=mix, x='genre', y='pct', hue='quarter',
            hue_order=quarter_order, palette=['#90CAF9', '#1565C0'], ax=ax)
ax.set_title('Genre Mix: Share of Sessions by Quarter', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('% of Sessions')
ax.legend(title='Quarter')
plt.tight_layout()
plt.show()

---

## Question 4: Predicting Completion

Using the features available in this dataset, build a model to predict whether a viewing session will result in a completed watch.

- What features would you use? Are there any you'd exclude and why?
- Choose an appropriate model and train it on Q4 2025 data
- Evaluate it on Q1 2026 data — what metrics matter here, and why?
- What does the model tell you about the most important drivers of completion?